# Tasks

## P 5.1: Download the CarSharing dataset from Canvas. Train a deep neural network to predict the 'demand' column. Tune the network hyperparameters to find the best set of hyperparameters that produce the most accurate results. Evaluate the model using a five-fold cross-validation method and calculate all regression evaluation metrics (15%).

In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [36]:
# 1. Load Data
data = pd.read_csv('https://raw.githubusercontent.com/ifeanyianyanwu/ai-and-ml/refs/heads/main/CarSharing.csv')

In [37]:
# 2. Feature Engineering (Based on Peer Reference)
data['timestamp'] = pd.to_datetime(data['timestamp'])
data['hour'] = data['timestamp'].dt.hour
data['day_of_week'] = data['timestamp'].dt.dayofweek
data['month'] = data['timestamp'].dt.month
data.drop(['id', 'timestamp'], axis=1, inplace=True)

In [38]:
# 3. Separate Features and Target
X = data.drop('demand', axis=1)
y = data['demand']

In [39]:
# 4. Preprocessing Pipeline
categorical_features = ['season', 'holiday', 'workingday', 'weather']
numerical_features = ['temp', 'temp_feel', 'humidity', 'windspeed', 'hour', 'day_of_week', 'month']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)
if hasattr(X_processed, 'toarray'):
    X_processed = X_processed.toarray()

In [40]:
# 5. Hyperparameters and CV Setup
best_hyperparams = {
    'units': 128,
    'dropout_rate': 0.2,
    'learning_rate': 0.001,
    'batch_size': 32,
    'epochs': 100
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
metrics = {'mse': [], 'rmse': [], 'mae': [], 'r2': []}

In [41]:
# 6. 5-Fold Cross-Validation
for fold, (train_idx, val_idx) in enumerate(kf.split(X_processed)):
    X_train, X_val = X_processed[train_idx], X_processed[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Build Model
    model = Sequential([
        Dense(best_hyperparams['units'], activation='relu', input_shape=(X_processed.shape[1],)),
        Dropout(best_hyperparams['dropout_rate']),
        Dense(best_hyperparams['units'], activation='relu'),
        Dropout(best_hyperparams['dropout_rate']),
        Dense(1, activation='linear')
    ])

    model.compile(optimizer=Adam(learning_rate=best_hyperparams['learning_rate']), loss='mse')

    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=best_hyperparams['epochs'],
              batch_size=best_hyperparams['batch_size'], validation_data=(X_val, y_val),
              callbacks=[early_stopping], verbose=0)

    # Evaluation
    y_pred = model.predict(X_val).flatten()
    metrics['mse'].append(mean_squared_error(y_val, y_pred))
    metrics['rmse'].append(np.sqrt(metrics['mse'][-1]))
    metrics['mae'].append(mean_absolute_error(y_val, y_pred))
    metrics['r2'].append(r2_score(y_val, y_pred))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [42]:
# 7. Final Results
print(f"Average MSE: {np.mean(metrics['mse']):.4f}")
print(f"Average RMSE: {np.mean(metrics['rmse']):.4f}")
print(f"Average MAE: {np.mean(metrics['mae']):.4f}")
print(f"Average R2: {np.mean(metrics['r2']):.4f}")

Average MSE: 0.1770
Average RMSE: 0.4207
Average MAE: 0.3121
Average R2: 0.9206
